In [1]:
!pip install openai

  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   -------- ------------------------------- 0.3/1.3 MB ? eta -:--:--
   -------------------------------- ------- 1.0/1.3 MB 3.7 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 3.5 MB/s  0:00:00
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   --------------------


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from dotenv import load_dotenv
import os
import json
import time

load_dotenv()  

from openai import OpenAI
client = OpenAI()  

print("Библиотеки загружены")

Библиотеки загружены


---
## Часть 0: Mock-инструменты (имитация внешних API)

In [3]:
def mock_get_weather(city: str) -> dict:
    """Возвращает mock-погоду для города."""
    result = {"city": city, "temp_c": 22, "condition": "Солнечно", "rain_chance": 5}
    print(f"  [TOOL: get_weather] {city} => {result['condition']}, {result['temp_c']}C")
    return result

def mock_search_attractions(city: str, category: str = "все") -> list:
    """Возвращает mock-достопримечательности по категории."""
    db = {
        "музеи": [
            {"name": "Лувр", "rating": 4.8, "type": "музей", "price_eur": 17},
            {"name": "Музей д'Орсе", "rating": 4.7, "type": "музей", "price_eur": 16},
            {"name": "Центр Помпиду", "rating": 4.5, "type": "музей", "price_eur": 15},
        ],
        "парки": [
            {"name": "Люксембургский сад", "rating": 4.8, "type": "парк", "price_eur": 0},
            {"name": "Булонский лес", "rating": 4.5, "type": "парк", "price_eur": 0},
        ],
        "архитектура": [
            {"name": "Эйфелева башня", "rating": 4.7, "type": "архитектура", "price_eur": 28},
            {"name": "Нотр-Дам де Пари", "rating": 4.6, "type": "архитектура", "price_eur": 0},
            {"name": "Версаль", "rating": 4.8, "type": "архитектура", "price_eur": 20},
        ],
    }
    if category in db:
        result = db[category]
    else:
        result = [item for sublist in db.values() for item in sublist][:5]
    print(f"  [TOOL: search_attractions] {city}, категория={category} => {len(result)} объектов")
    return result

def mock_search_flights(origin: str, destination: str, date: str = "2025-09-15") -> list:
    """Возвращает mock-рейсы."""
    flights = [
        {"airline": "Aeroflot", "flight": "SU2454", "departure": "08:30", "price_rub": 24500},
        {"airline": "Air France", "flight": "AF1655", "departure": "14:20", "price_rub": 31200},
        {"airline": "S7 Airlines", "flight": "S72106", "departure": "21:00", "price_rub": 19800},
    ]
    print(f"  [TOOL: search_flights] {origin} -> {destination} ({date}) => {len(flights)} рейсов")
    return flights

def mock_search_hotels(city: str, budget: str = "средний") -> list:
    """Возвращает mock-отели."""
    db = {
        "эконом": [{"name": "Generator Paris", "stars": 3, "price_per_night_eur": 45}],
        "средний": [
            {"name": "Hotel des Grands Boulevards", "stars": 4, "price_per_night_eur": 180},
            {"name": "Hotel National des Arts et Metiers", "stars": 4, "price_per_night_eur": 165},
        ],
        "люкс": [{"name": "Le Meurice", "stars": 5, "price_per_night_eur": 900}],
    }
    result = db.get(budget, db["средний"])
    print(f"  [TOOL: search_hotels] {city}, бюджет={budget} => {len(result)} отелей")
    return result

print("Mock-инструменты определены")

Mock-инструменты определены


---
## Часть 1: Жёсткий Воркфлоу (Hard-Coded Workflow)

Поток выполнения **всегда фиксирован**: погода → рейсы → отели → достопримечательности → генерация текста.  
Разработчик — архитектор потока. LLM вызывается только **один раз** для финальной генерации.

In [5]:
class TravelWorkflow:
    """
    Детерминированный конвейер планирования поездки.
    Шаги выполняются ВСЕГДА в одном порядке, независимо от запроса.
    """

    def __init__(self, model: str = "gpt-4o-mini"):
        self.model = model
        self.token_usage = {"input": 0, "output": 0}
        self.steps_log = []

    def _llm_call(self, prompt: str) -> str:
        response = client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
        )
        self.token_usage["input"] += response.usage.prompt_tokens
        self.token_usage["output"] += response.usage.completion_tokens
        return response.choices[0].message.content

    def run(self, user_query: str, origin: str, destination: str) -> str:
        print("=" * 60)
        print("ЗАПУСК WORKFLOW")
        print(f"Запрос: {user_query}")
        print("=" * 60)
        start = time.time()

        print("\n[ШАГ 1] Погода")
        weather = mock_get_weather(destination)

        print("\n[ШАГ 2] Рейсы")
        flights = mock_search_flights(origin, destination)

        print("\n[ШАГ 3] Отели")
        hotels = mock_search_hotels(destination)

        print("\n[ШАГ 4] Достопримечательности (категория='все' — жёстко задана)")
        sights = mock_search_attractions(destination, "все")

        print("\n[ШАГ 5] Генерация итогового плана через LLM")
        prompt = f"""Пользователь спрашивает: {user_query}

Данные:
- Погода: {json.dumps(weather, ensure_ascii=False)}
- Лучший рейс: {json.dumps(flights[0], ensure_ascii=False)}
- Рекомендуемый отель: {json.dumps(hotels[0], ensure_ascii=False)}
- Достопримечательности: {json.dumps(sights[:4], ensure_ascii=False)}

Составь краткий план поездки на 3 дня на русском языке. Будь конкретным."""
        plan = self._llm_call(prompt)

        elapsed = time.time() - start
        print("\n" + "=" * 60)
        print("СТАТИСТИКА WORKFLOW")
        print(f"  Время        : {elapsed:.2f} сек")
        print(f"  Токены вход  : {self.token_usage['input']}")
        print(f"  Токены выход : {self.token_usage['output']}")
        print(f"  Вызовов LLM  : 1")
        print("=" * 60)
        return plan

In [6]:
workflow = TravelWorkflow(model="gpt-4o-mini")
workflow_result = workflow.run(
    user_query="Хочу поехать в Париж на 3 дня. Что посмотреть, какая погода и как добраться из Москвы?",
    origin="Москва",
    destination="Париж",
)

print("\nИТОГОВЫЙ ПЛАН (WORKFLOW):")
print(workflow_result)

ЗАПУСК WORKFLOW
Запрос: Хочу поехать в Париж на 3 дня. Что посмотреть, какая погода и как добраться из Москвы?

[ШАГ 1] Погода
  [TOOL: get_weather] Париж => Солнечно, 22C

[ШАГ 2] Рейсы
  [TOOL: search_flights] Москва -> Париж (2025-09-15) => 3 рейсов

[ШАГ 3] Отели
  [TOOL: search_hotels] Париж, бюджет=средний => 2 отелей

[ШАГ 4] Достопримечательности (категория='все' — жёстко задана)
  [TOOL: search_attractions] Париж, категория=все => 5 объектов

[ШАГ 5] Генерация итогового плана через LLM

СТАТИСТИКА WORKFLOW
  Время        : 25.53 сек
  Токены вход  : 317
  Токены выход : 434
  Вызовов LLM  : 1

ИТОГОВЫЙ ПЛАН (WORKFLOW):
### План поездки в Париж на 3 дня

#### Погода
- **Температура:** 22°C
- **Условия:** Солнечно
- **Вероятность дождя:** 5%

#### Как добраться из Москвы
- **Авиакомпания:** Aeroflot
- **Рейс:** SU2454
- **Время вылета:** 08:30
- **Цена билета:** 24,500 руб.

#### Рекомендуемый отель
- **Название:** Hotel des Grands Boulevards
- **Звезды:** 4
- **Цена за ночь:** 

---
## Часть 2: Автономный Агент (ReAct Agent)

Здесь **LLM сам решает** какие инструменты вызывать, в каком порядке и сколько раз.  
Паттерн **ReAct**: Think → Act → Observe → Repeat до готовности.

In [7]:
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Получить текущую погоду в указанном городе",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_attractions",
            "description": "Найти достопримечательности. Категории: музеи, парки, архитектура, все",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"},
                    "category": {
                        "type": "string",
                        "enum": ["музеи", "парки", "архитектура", "все"],
                    },
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_flights",
            "description": "Найти авиарейсы из города в город",
            "parameters": {
                "type": "object",
                "properties": {
                    "origin": {"type": "string"},
                    "destination": {"type": "string"},
                    "date": {"type": "string"},
                },
                "required": ["origin", "destination"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_hotels",
            "description": "Найти отели. Бюджет: эконом, средний, люкс",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"},
                    "budget": {
                        "type": "string",
                        "enum": ["эконом", "средний", "люкс"],
                    },
                },
                "required": ["city"],
            },
        },
    },
]

def execute_tool(name: str, args: dict) -> str:
    dispatch = {
        "get_weather": mock_get_weather,
        "search_attractions": mock_search_attractions,
        "search_flights": mock_search_flights,
        "search_hotels": mock_search_hotels,
    }
    fn = dispatch.get(name)
    if not fn:
        return json.dumps({"error": f"Неизвестный инструмент: {name}"})
    return json.dumps(fn(**args), ensure_ascii=False)

print("Схемы инструментов агента определены")

Схемы инструментов агента определены


In [9]:
class TravelAgent:
    """
    Автономный агент на основе ReAct (Function Calling OpenAI).
    LLM САМА решает: какие инструменты вызывать, сколько раз, в каком порядке.
    """

    SYSTEM_PROMPT = """Ты — опытный турагент. Помоги пользователю спланировать поездку.

Инструменты: get_weather, search_flights, search_hotels, search_attractions (категории: музеи/парки/архитектура/все).

Правила:
- Если погода плохая (дождь) — предлагай музеи и крытые места, не парки.
- Используй ТОЛЬКО нужные инструменты для конкретного запроса.
- Составь подробный итоговый план на основе собранных данных."""

    def __init__(self, model: str = "gpt-4o-mini", max_iterations: int = 10):
        self.model = model
        self.max_iterations = max_iterations
        self.token_usage = {"input": 0, "output": 0}
        self.thought_log = []
        self.tool_calls_count = 0
        self.llm_calls_count = 0

    def run(self, user_query: str) -> str:
        print("=" * 60)
        print("ЗАПУСК АВТОНОМНОГО АГЕНТА (ReAct)")
        print(f"Запрос: {user_query}")
        print("=" * 60)
        start = time.time()

        messages = [
            {"role": "system", "content": self.SYSTEM_PROMPT},
            {"role": "user", "content": user_query},
        ]

        for iteration in range(1, self.max_iterations + 1):
            print(f"\n--- Итерация ReAct #{iteration} ---")

            self.llm_calls_count += 1
            response = client.chat.completions.create(
                model=self.model,
                messages=messages,
                tools=TOOLS_SCHEMA,
                tool_choice="auto",
                temperature=0.2,
            )
            self.token_usage["input"] += response.usage.prompt_tokens
            self.token_usage["output"] += response.usage.completion_tokens

            msg = response.choices[0].message
            messages.append(msg)

            finish = response.choices[0].finish_reason
            self.thought_log.append({"iteration": iteration, "finish_reason": finish})

            if msg.content:
                print(f"  [THINK] {msg.content[:300]}")

            if finish == "stop" or not msg.tool_calls:
                print("  [STOP] Агент завершил работу")
                break

            for tc in msg.tool_calls:
                self.tool_calls_count += 1
                name = tc.function.name
                args = json.loads(tc.function.arguments)
                print(f"  [ACT] {name}({args})")

                result = execute_tool(name, args)
                messages.append(
                    {"role": "tool", "tool_call_id": tc.id, "content": result}
                )

        final = ""
        for m in reversed(messages):
            if hasattr(m, "role") and m.role == "assistant" and m.content:
                if not (hasattr(m, "tool_calls") and m.tool_calls):
                    final = m.content
                    break

        elapsed = time.time() - start
        print("\n" + "=" * 60)
        print("СТАТИСТИКА АГЕНТА")
        print(f"  Время          : {elapsed:.2f} сек")
        print(f"  Токены вход    : {self.token_usage['input']}")
        print(f"  Токены выход   : {self.token_usage['output']}")
        print(f"  Итераций ReAct : {len(self.thought_log)}")
        print(f"  Вызовов LLM    : {self.llm_calls_count}")
        print(f"  Вызовов инстр. : {self.tool_calls_count}")
        print("=" * 60)
        return final

In [10]:
agent = TravelAgent(model="gpt-4o-mini")
agent_result = agent.run(
    "Хочу поехать в Париж на 3 дня. Что посмотреть, какая погода и как добраться из Москвы?"
)

print("\nИТОГОВЫЙ ПЛАН (АГЕНТ):")
print(agent_result)

ЗАПУСК АВТОНОМНОГО АГЕНТА (ReAct)
Запрос: Хочу поехать в Париж на 3 дня. Что посмотреть, какая погода и как добраться из Москвы?

--- Итерация ReAct #1 ---
  [ACT] get_weather({'city': 'Париж'})
  [TOOL: get_weather] Париж => Солнечно, 22C
  [ACT] search_flights({'origin': 'Москва', 'destination': 'Париж'})
  [TOOL: search_flights] Москва -> Париж (2025-09-15) => 3 рейсов
  [ACT] search_attractions({'city': 'Париж', 'category': 'все'})
  [TOOL: search_attractions] Париж, категория=все => 5 объектов

--- Итерация ReAct #2 ---
  [THINK] Вот план вашей поездки в Париж на 3 дня:

### Погода в Париже
- Температура: 22°C
- Условия: Солнечно
- Вероятность дождя: 5%

### Как добраться из Москвы в Париж
Вот несколько вариантов рейсов:
1. **Aeroflot**
   - Рейс: SU2454
   - Время вылета: 08:30
   - Цена: 24,500 руб.
   
2. **Air France**
 
  [STOP] Агент завершил работу

СТАТИСТИКА АГЕНТА
  Время          : 12.55 сек
  Токены вход    : 1059
  Токены выход   : 549
  Итераций ReAct : 2
  Вызовов L

---
## Часть 3: Тест на нестандартный запрос

> «У меня ограниченный бюджет и я боюсь летать — только поезд. Хочу посетить только музеи.»

**Ожидаемое поведение:**
- **Workflow** — выполнит все 5 шагов включая поиск рейсов (ненужных!) и найдёт все категории мест
- **Агент** — адаптируется: пропустит search_flights, вызовет search_hotels(эконом), search_attractions(музеи)

In [11]:
UNUSUAL = "У меня ограниченный бюджет и я боюсь летать — только поезд. Хочу посетить только музеи Парижа."

print("=" * 60)
print("ТЕСТ: НЕСТАНДАРТНЫЙ ЗАПРОС")
print("=" * 60)

print("\n>> WORKFLOW (ожидаем лишние шаги):")
wf2 = TravelWorkflow()
wf2.run(user_query=UNUSUAL, origin="Москва", destination="Париж")

print("\n>> АГЕНТ (ожидаем адаптацию):")
ag2 = TravelAgent()
ag2.run(UNUSUAL)

print(f"\nWorkflow: вызвал search_flights? ВСЕГДА ДА (жёстко задано)")
print(f"Агент:    вызвал search_flights? {'ДА' if any(t.get('finish_reason') for t in ag2.thought_log) else 'ПРОВЕРЬТЕ ЛОГИ'}")
print(f"Агент:    всего инструментов вызвано = {ag2.tool_calls_count} (меньше = лучше адаптировался)")

ТЕСТ: НЕСТАНДАРТНЫЙ ЗАПРОС

>> WORKFLOW (ожидаем лишние шаги):
ЗАПУСК WORKFLOW
Запрос: У меня ограниченный бюджет и я боюсь летать — только поезд. Хочу посетить только музеи Парижа.

[ШАГ 1] Погода
  [TOOL: get_weather] Париж => Солнечно, 22C

[ШАГ 2] Рейсы
  [TOOL: search_flights] Москва -> Париж (2025-09-15) => 3 рейсов

[ШАГ 3] Отели
  [TOOL: search_hotels] Париж, бюджет=средний => 2 отелей

[ШАГ 4] Достопримечательности (категория='все' — жёстко задана)
  [TOOL: search_attractions] Париж, категория=все => 5 объектов

[ШАГ 5] Генерация итогового плана через LLM

СТАТИСТИКА WORKFLOW
  Время        : 6.40 сек
  Токены вход  : 317
  Токены выход : 384
  Вызовов LLM  : 1

>> АГЕНТ (ожидаем адаптацию):
ЗАПУСК АВТОНОМНОГО АГЕНТА (ReAct)
Запрос: У меня ограниченный бюджет и я боюсь летать — только поезд. Хочу посетить только музеи Парижа.

--- Итерация ReAct #1 ---
  [ACT] get_weather({'city': 'Париж'})
  [TOOL: get_weather] Париж => Солнечно, 22C

--- Итерация ReAct #2 ---
  [ACT] search_

---
## Часть 4: Итоговое сравнение

In [12]:
rows = [
    ("Предсказуемость",
     "ВЫСОКАЯ — один путь всегда",
     "НИЗКАЯ — зависит от LLM и промпта"),
    ("Гибкость",
     "НИЗКАЯ — нельзя пропустить шаги",
     "ВЫСОКАЯ — адаптируется к запросу"),
    ("Токены (стоимость)",
     "ЭКОНОМНО — LLM 1 раз",
     "ДОРОЖЕ — LLM N итераций"),
    ("Скорость (latency)",
     "БЫСТРО — нет цикла мышления",
     "МЕДЛЕННЕЕ — каждый шаг ждёт LLM"),
    ("Сложность отладки",
     "ЛЕГКО — trace линейный",
     "СЛОЖНЕЕ — нужен анализ thoughts"),
    ("Нестандартный запрос",
     "Выполнит лишние шаги",
     "Пропустит ненужное"),
]

print(f"{'Критерий':<25} {'Workflow':<40} {'Agent':<40}")
print("-" * 105)
for r in rows:
    print(f"{r[0]:<25} {r[1]:<40} {r[2]:<40}")

print("\nВЫВОДЫ:")
print("Используй Workflow когда: фиксированный процесс, продакшн, важна надёжность и стоимость")
print("Используй Агента когда: разнообразные запросы, неизвестна последовательность шагов, важна гибкость")

Критерий                  Workflow                                 Agent                                   
---------------------------------------------------------------------------------------------------------
Предсказуемость           ВЫСОКАЯ — один путь всегда               НИЗКАЯ — зависит от LLM и промпта       
Гибкость                  НИЗКАЯ — нельзя пропустить шаги          ВЫСОКАЯ — адаптируется к запросу        
Токены (стоимость)        ЭКОНОМНО — LLM 1 раз                     ДОРОЖЕ — LLM N итераций                 
Скорость (latency)        БЫСТРО — нет цикла мышления              МЕДЛЕННЕЕ — каждый шаг ждёт LLM         
Сложность отладки         ЛЕГКО — trace линейный                   СЛОЖНЕЕ — нужен анализ thoughts         
Нестандартный запрос      Выполнит лишние шаги                     Пропустит ненужное                      

ВЫВОДЫ:
Используй Workflow когда: фиксированный процесс, продакшн, важна надёжность и стоимость
Используй Агента когда: разнообразные зап